### Data Processing and Transformation
We will load and clean the selected files. For the Unified Transaction CSV, we will skip the first 13 rows.

In [1]:
import pandas as pd
import os

# Define file paths
txt_file = './content/154442683841020730.txt'
transaction_file = './content/2026Aug1-2026Sep30CustomUnifiedTransaction.csv'
tsv_file = './content/report-50212020730.tsv'
sku_master = './content/SKU Master.xlsx'

# 1. Process and convert the TXT file to CSV
try:
    # Typically, these text files from reports are tab-separated or comma-separated.
    # We will attempt reading as tab-separated first. If it has a different format, pandas handles it.
    df_txt = pd.read_csv(txt_file, sep='\t', encoding='ISO-8859-1')
    df_txt.columns = df_txt.columns.str.strip()
    txt_output_path = './content/processed_text_file.csv'
    df_txt.to_csv(txt_output_path, index=False)
    print(f"Successfully processed TXT file and saved to CSV: {txt_output_path} (Shape: {df_txt.shape})")
except Exception as e:
    print("Error processing TXT file:", e)

# 2. Process Unified Transaction file (skipping first 13 rows)
try:
    df_transaction = pd.read_csv(transaction_file, skiprows=13)
    df_transaction.columns = df_transaction.columns.str.strip()
    df_transaction['total'] = df_transaction['total'].astype(str).str.replace(',', '').astype(float)
    transaction_output_path = './content/processed_unified_transaction.csv'
    df_transaction.to_csv(transaction_output_path, index=False)
    print(f"Successfully processed Transaction file and saved to CSV: {transaction_output_path} (Shape: {df_transaction.shape})")
except Exception as e:
    print("Error processing Transaction file:", e)

# 3. Process TSV File
try:
    df_tsv = pd.read_csv(tsv_file, sep='\t', encoding='ISO-8859-1')
    df_tsv.columns = df_tsv.columns.str.strip()
    tsv_output_path = '/content/processed_tsv.csv'
    df_tsv.to_csv(tsv_output_path, index=False)
    print(f"Successfully processed TSV file and saved to CSV: {tsv_output_path} (Shape: {df_tsv.shape})")
except Exception as e:
    print("Error processing TSV file:", e)

Successfully processed TXT file and saved to CSV: ./content/processed_text_file.csv (Shape: (886, 35))
Successfully processed Transaction file and saved to CSV: ./content/processed_unified_transaction.csv (Shape: (3588, 29))
Error processing TSV file: Cannot save file into a non-existent directory: '/content'


In [2]:
# Clean column names to make sure hyphens and spaces match correctly
df_txt.columns = df_txt.columns.str.strip().str.replace(' ', '-')

# List of requested columns
columns_to_extract = [
    'amazon-order-id',
    'sku',
    'quantity',
    'item-price',
]

# Create df_orders containing only the requested columns
df_orders = df_txt[columns_to_extract].copy()

# Display information and the first few rows of the new DataFrame
print("df_orders Shape:", df_orders.shape)
display(df_orders.head())

df_orders Shape: (886, 4)


,amazon-order-id,sku,quantity,item-price
0,171-0152343-0393937,G-HIVE-CURVE PLANT SUPPORT-XL-2PC,1,999.0
1,407-3305021-9963562,G-HIVE-CURVE PLANT SUPPORT,1,279.0
2,404-6311479-6709112,G-HIVE-Garden Hose Holder,1,299.0
3,407-7159209-0411503,G-HIVE-STAR PLANT SUPPORT,1,180.0
4,404-5324137-3896354,G-HIVE-STAR PLANT SUPPORT,1,180.0


In [3]:
# Drop rows where quantity is 0 or item-price is NaN or 0
df_orders = df_orders[
    (df_orders['quantity'] != 0) &
    (df_orders['item-price'].notna()) &
    (df_orders['item-price'] != 0)
]

# Display the new DataFrame size and first few rows
print("df_orders shape after filtering:", df_orders.shape)
display(df_orders.head())

df_orders shape after filtering: (836, 4)


,amazon-order-id,sku,quantity,item-price
0,171-0152343-0393937,G-HIVE-CURVE PLANT SUPPORT-XL-2PC,1,999.0
1,407-3305021-9963562,G-HIVE-CURVE PLANT SUPPORT,1,279.0
2,404-6311479-6709112,G-HIVE-Garden Hose Holder,1,299.0
3,407-7159209-0411503,G-HIVE-STAR PLANT SUPPORT,1,180.0
4,404-5324137-3896354,G-HIVE-STAR PLANT SUPPORT,1,180.0


In [4]:
columns_to_extract = [
    'Order ID',
    'Merchant SKU',
    'Return type',
]

df_order_returns = df_tsv[columns_to_extract].copy()
df_order_returns['Order ID'] = df_order_returns['Order ID'].astype('string').str.strip()
df_order_returns['Merchant SKU'] = df_order_returns['Merchant SKU'].astype('string').str.strip().str.casefold()

df_order_returns = df_order_returns.dropna(subset=['Order ID', 'Merchant SKU'])
df_order_returns = df_order_returns[
    df_order_returns['Order ID'].ne('') & df_order_returns['Merchant SKU'].ne('')
]

print("df_order_returns Shape:", df_order_returns.shape)
display(df_order_returns.head())

df_order_returns Shape: (151, 3)


,Order ID,Merchant SKU,Return type
0,408-0050973-1445976,g-hive-ties,Amazon CS
1,402-6338036-7377948,g-hive-star plant support-16pcs,C-Returns
2,403-9486989-3003530,g-hive-plant-tie-strap -12 roll,Rejected
3,406-8312913-2016307,g-hive-curve plant support-xl,Rejected
4,404-5324405-8881115,g-hive-curve plant support-xl,C-Returns


In [5]:
# Match returns to individual order/SKU pairs; a set naturally removes duplicate report rows.
returned_order_skus = set(
    zip(df_order_returns['Order ID'], df_order_returns['Merchant SKU'])
)

# Create a new DataFrame as a copy of df_orders
df_orders_with_returns = df_orders.copy()
order_keys = df_orders_with_returns['amazon-order-id'].astype('string').str.strip()
sku_keys = df_orders_with_returns['sku'].astype('string').str.strip().str.casefold()

# Mark only the product lines whose order ID and SKU both appear in the returns report.
df_orders_with_returns['is_returned'] = [
    int((order_id, sku) in returned_order_skus)
    for order_id, sku in zip(order_keys, sku_keys)
]

# Display the value counts of the returns and the first few rows of the new DataFrame
print("Value counts for 'is_returned':")
print(df_orders_with_returns['is_returned'].value_counts())
print("\nNew DataFrame shape:", df_orders_with_returns.shape)
display(df_orders_with_returns.sample(5))

Value counts for 'is_returned':
is_returned
0    749
1     87
Name: count, dtype: int64

New DataFrame shape: (836, 5)


,amazon-order-id,sku,quantity,item-price,is_returned
16,408-7222644-5901931,G-HIVE-STAR PLANT SUPPORT,1,180.0,0
863,405-7929723-7153903,G-HIVE-STAR PLANT SUPPORT-8PCS,1,255.0,1
237,403-6754459-9141948,G-HIVE-STAR PLANT SUPPORT,1,180.0,0
667,403-5813018-9755503,G-HIVE-TIES,1,175.0,1
782,403-7984936-2254749,G-HIVE-TIES-60,5,1425.0,1


In [6]:
import pandas as pd

# Load the SKU Master excel file
sku_master_file = './content/SKU Master.xlsx'
try:
    df_sku_master = pd.read_excel(sku_master_file)
    # Clean column names by stripping spaces
    df_sku_master.columns = df_sku_master.columns.str.strip()
    # Clean text columns in the data itself
    if 'SKU' in df_sku_master.columns:
        df_sku_master['SKU'] = df_sku_master['SKU'].astype(str).str.strip()
    print("SKU Master Shape:", df_sku_master.shape)
    print("Columns:", list(df_sku_master.columns))
    display(df_sku_master.head())
except Exception as e:
    print("Error loading SKU Master file:", e)

SKU Master Shape: (43, 2)
Columns: ['SKU', 'Cost']


,SKU,Cost
0,Full Spectrun light with wodden base,900
1,G-HIVE- Cartoon Chicken Feet,100
2,G-HIVE- Coffee Cup Table,110
3,G-HIVE- Coffee Cup Table-Gold,110
4,G-HIVE- Metal Planter Stand Garden-Gold,100


In [7]:
columns_to_extract = [
    'Sku',
    'type',
    'order id',
    'quantity',
    'total'
]

df_transaction = df_transaction[columns_to_extract].copy()

print("df_transaction Shape:", df_transaction.shape)
display(df_transaction.sample(5))


df_transaction Shape: (3588, 5)


,Sku,type,order id,quantity,total
2788,G-HIVE-TIES,Order,402-3455706-9054724,1.0,171.75
2261,G-HIVE-CURVE PLANT SUPPORT,Order,408-1693507-3049104,1.0,269.66
928,G-HIVE-TIES-40,Order,403-0844332-3073122,1.0,216.70
2715,G-HIVE-STAR PLANT SUPPORT,Order,171-0951225-1268365,1.0,176.72
1140,NaN,Shipping Services,403-6782357-4789907,NaN,-64.90


In [8]:
import pandas as pd
import numpy as np

# Work on a copy of df_transaction to preserve raw data
trans_df = df_transaction.copy()

# Ensure clean strings for order id
trans_df['order id'] = trans_df['order id'].astype(str).str.strip()

# Convert 'total' column to numeric
trans_df['total'] = pd.to_numeric(trans_df['total'], errors='coerce').fillna(0)

# If order id is missing, NaN, or 'nan', fill it with 'others'
trans_df['order id'] = trans_df['order id'].replace({'nan': 'others', '': 'others'})
trans_df['order id'] = trans_df['order id'].fillna('others')

# Group strictly by 'order id' and sum the total payouts (excluding Sku entirely as requested)
order_grouped = trans_df.groupby('order id').agg({
    'total': 'sum'
}).reset_index()

# Rename total to Net_Payout
order_grouped.rename(columns={'total': 'Net_Payout'}, inplace=True)

print("Grouped Transactions shape:", order_grouped.shape)
display(order_grouped.sample(10))
print("\n--- 'others' (No Order ID) Summary ---")
display(order_grouped[order_grouped['order id'] == 'others'])

print()
print("Sum of payouts for valid Order IDs:")
print(
    order_grouped[order_grouped['order id'] != 'others'].sum(numeric_only=True)
)

Grouped Transactions shape: (1664, 2)


,order id,Net_Payout
409,402-9376934-3392340,111.82
620,403-9226641-8369111,210.66
41,171-1506092-8336350,221.45
566,403-6782357-4789907,192.29
998,405-8124420-0577901,111.82
1459,407-9352254-5299523,111.82
1442,407-8438896-2907506,113.00
240,402-1419379-4685956,111.82
289,402-3724356-8610721,197.56
522,403-5282443-8117111,196.39



--- 'others' (No Order ID) Summary ---


,order id,Net_Payout
1663,others,-281182.33



Sum of payouts for valid Order IDs:
Net_Payout    313045.25
dtype: float64


In [9]:
import pandas as pd
import numpy as np

# 1. Prepare Orders and map unit cost
orders = df_orders_with_returns.copy()
orders['sku'] = orders['sku'].astype(str).str.strip()
orders['amazon-order-id'] = orders['amazon-order-id'].astype(str).str.strip()

sku_costs = df_sku_master.copy()
sku_costs['SKU'] = sku_costs['SKU'].astype(str).str.strip()
s_costs = sku_costs.rename(columns={'SKU': 'sku', 'Cost': 'Unit_Cost'})

# Keep all 836 rows by merging on SKU
orders_with_cost = pd.merge(orders, s_costs, on='sku', how='left')
orders_with_cost['Unit_Cost'] = orders_with_cost['Unit_Cost'].fillna(0)

# 2. Map Net Payment for each specific order from transaction table
trans_payouts = order_grouped[order_grouped['order id'] != 'others'].rename(columns={'order id': 'amazon-order-id'})

# To avoid duplicating payouts when an order has multiple SKUs, we can allocate the payout or map it.
# Since we are keeping 836 unique (order-id, sku) combinations separate, we left-merge the payout.
# Note: To avoid double counting total payouts if multiple SKUs share the same order-id when summing total payouts,
# we will calculate payouts correctly by ensuring each order id's payout is counted only once.
final_reconciliation = pd.merge(orders_with_cost, trans_payouts[['amazon-order-id', 'Net_Payout']], on='amazon-order-id', how='left')
final_reconciliation['Net_Payout'] = final_reconciliation['Net_Payout'].fillna(0)

# 3. Calculate COGS (only applies to non-returned products, multiplying Unit Cost * Quantity)
def calculate_cogs(row):
    if row['is_returned'] == 1:
        # Product returned: cost is recovered
        return 0.0
    else:
        # Product delivered successfully: Cost = Unit Cost * Ordered Quantity
        return float(row['quantity'] * row['Unit_Cost'])

final_reconciliation['COGS'] = final_reconciliation.apply(calculate_cogs, axis=1)

# 4. Aggregated Summary while preserving the 836 lines
total_lines_count = len(final_reconciliation)

# Calculate true payouts by getting unique payouts per order id to prevent double-counting due to splits
unique_order_payouts = final_reconciliation.drop_duplicates(subset=['amazon-order-id'])
total_net_payout = unique_order_payouts['Net_Payout'].sum()

total_cogs = final_reconciliation['COGS'].sum()

# Packaging Cost = flat avg packing cost (15) * total number of unique orders
PACKAGING_COST_PER_ORDER = 15
total_orders_count = final_reconciliation['amazon-order-id'].nunique()
total_packaging_cost = total_orders_count * PACKAGING_COST_PER_ORDER

print("--- Reconciled Monthly Performance Summary ---")
print(f"Total Item Rows (Excel Lines): {total_lines_count}")
print(f"Unique Orders in Month:        {total_orders_count}")
print(f"Total Net Order Payouts:       {total_net_payout:,.2f} RS")
print(f"Total Cost of Goods (COGS):     {total_cogs:,.2f} RS")
print(f"Total Packaging Cost (15*N):    {total_packaging_cost:,.2f} RS\n")

# 5. Inputs for Ad Expense and Other Expenses
try:
    ad_exp_input = input("Enter Monthly Ad Expense in RS: ")
    ad_expenses = float(ad_exp_input) if ad_exp_input.strip() != '' else 0.0
except ValueError:
    print("Invalid input. Defaulting Ad expenses to 0.0 RS.")
    ad_expenses = 0.0

try:
    other_exp_input = input("Enter Other Miscellaneous Expense in RS: ")
    other_expenses = float(other_exp_input) if other_exp_input.strip() != '' else 0.0
except ValueError:
    print("Invalid input. Defaulting Other expenses to 0.0 RS.")
    other_expenses = 0.0

# Calculate final net profit/loss
final_net_profit = total_net_payout - total_cogs - total_packaging_cost - ad_expenses - other_expenses
print(f"\nFinal Net Profit / Loss: {final_net_profit:,.2f} RS")

--- Reconciled Monthly Performance Summary ---
Total Item Rows (Excel Lines): 836
Unique Orders in Month:        826
Total Net Order Payouts:       142,139.88 RS
Total Cost of Goods (COGS):     72,935.00 RS
Total Packaging Cost (15*N):    12,390.00 RS




Final Net Profit / Loss: 1,314.88 RS


In [10]:
import pandas as pd

# Export the final_reconciliation table to a CSV file
output_csv_path = './content/final_reconciliation.xlsx'
final_reconciliation.to_excel(output_csv_path, index=False)
print(f"Successfully exported final_reconciliation table to: {output_csv_path}")


Successfully exported final_reconciliation table to: ./content/final_reconciliation.xlsx


In [11]:
# Find duplicate order IDs in your orders dataset
duplicate_mask = df_orders_with_returns.duplicated(subset=['amazon-order-id'], keep=False)
duplicate_orders = df_orders_with_returns[duplicate_mask].sort_values(by='amazon-order-id')

print(f"Total duplicated rows in orders file: {len(duplicate_orders)}")
print(f"These rows belong to {duplicate_orders['amazon-order-id'].nunique()} unique multi-item orders.\n")

print("Here are some examples of orders that have multiple rows (different SKUs or splits) in your Excel:")
display(duplicate_orders.head(10))

Total duplicated rows in orders file: 20
These rows belong to 10 unique multi-item orders.

Here are some examples of orders that have multiple rows (different SKUs or splits) in your Excel:


,amazon-order-id,sku,quantity,item-price,is_returned
96,171-8173305-4428318,G-HIVE-TOMATO PLANT CLIPS,1,229.0,0
97,171-8173305-4428318,G-HIVE-STAR PLANT SUPPORT-16PCS,1,399.0,0
88,171-8207199-5786761,G-HIVE-CURVE PLANT SUPPORT-XL,1,549.0,0
89,171-8207199-5786761,G-HIVE-CURVE PLANT SUPPORT-5mm -2PCS,1,659.0,0
285,403-3238301-0033110,Tabletop Planter Stand,1,419.0,1
284,403-3238301-0033110,G-HIVE-Unique Triangle Shape-Gold,1,419.0,1
49,405-9360423-3765939,G-HIVE-CURVE PLANT SUPPORT-XL,1,549.0,1
50,405-9360423-3765939,G-HIVE-CURVE PLANT SUPPORT,1,279.0,1
456,406-1664349-2062716,G-HIVE-CURVE PLANT SUPPORT,1,279.0,0
455,406-1664349-2062716,G-HIVE-CURVE PLANT SUPPORT-5mm,1,399.0,0


In [12]:
import pandas as pd
import numpy as np

# Work on a copy of the 836-row reconciliation table
base_df = final_reconciliation.copy()

# Clean keys
base_df['amazon-order-id'] = base_df['amazon-order-id'].astype(str).str.strip()
base_df['sku'] = base_df['sku'].astype(str).str.strip()

# Custom aggregation logic
def aggregate_unique_orders(group):
    # 1. Unique order id is the group key
    order_id = group.name

    # 2. Net payout is unique per order (all rows in same order share the same payout)
    net_payout = group['Net_Payout'].iloc[0]

    # 3. is_returned is 1 if any item in the order is returned, else 0
    is_returned = int(group['is_returned'].max())

    # 4. Quantity string in format: (qty1 + qty2 + ...)
    qty_list = group['quantity'].astype(int).astype(str).tolist()
    qty_format = " + ".join(qty_list)

    # 5. Cost price formula string: (qty1 * cp1 + qty2 * cp2 + ...)
    cp_elements = []
    for _, row in group.iterrows():
        cp_elements.append(f"{int(row['quantity'])} * {row['Unit_Cost']:.1f}")
    cost_price_format = " + ".join(cp_elements)

    # 6. Final total COGS (sum of calculated cogs for each row)
    total_cogs = group['COGS'].sum()

    return pd.Series({
        'amazon-order-id': order_id,
        'Net_Payout': net_payout,
        'is_returned': is_returned,
        'quantity_format': qty_format,
        'cost_price_breakdown': cost_price_format,
        'Total_COGS': total_cogs
    })

# Group and apply custom aggregator
reconciliation_unique = base_df.groupby('amazon-order-id', as_index=False).apply(aggregate_unique_orders)

# Reset index and verify shapes
reconciliation_unique.reset_index(drop=True, inplace=True)
print("Reconciliation Unique shape:", reconciliation_unique.shape)
print(f"Total unique order IDs: {reconciliation_unique['amazon-order-id'].nunique()}")

# Export to Excel format
output_xlsx = './content/final_reconciliation_unique.xlsx'
reconciliation_unique.to_excel(output_xlsx, index=False)
print(f"Successfully exported custom unique order sheet to: {output_xlsx}")

# Display sample of multi-item orders if any, else head
multi_items = reconciliation_unique[reconciliation_unique['quantity_format'].str.contains('\\+')]
if not multi_items.empty:
    print("\n--- Sample of Multi-item Orders in the customized format ---")
    display(multi_items.head(5))
else:
    print("\n--- Sample output ---")
    display(reconciliation_unique.head(5))


Reconciliation Unique shape: (826, 6)
Total unique order IDs: 826
Successfully exported custom unique order sheet to: ./content/final_reconciliation_unique.xlsx

--- Sample of Multi-item Orders in the customized format ---


,amazon-order-id,Net_Payout,is_returned,quantity_format,cost_price_breakdown,Total_COGS
78,171-8173305-4428318,501.51,0,1 + 1,1 * 100.0 + 1 * 160.0,260.0
79,171-8207199-5786761,906.31,0,1 + 1,1 * 240.0 + 1 * 300.0,540.0
235,403-3238301-0033110,-390.08,1,1 + 1,1 * 100.0 + 1 * 100.0,0.0
509,405-9360423-3765939,86.69,1,1 + 1,1 * 240.0 + 1 * 105.0,0.0
534,406-1664349-2062716,525.17,0,1 + 1,1 * 150.0 + 1 * 105.0,255.0
